# ProteinMPNN execution example
Design optimal sequences based on protein 3D structures.
## Overview
- **Field** — Protein design
- **Category** — Protein Design
- **Model type** — MPNN (GNN)
- **Year** — 2022
- **License** — MIT
**Related papers**

- [Robust deep learning based protein sequence design using ProteinMPNN](https://www.biorxiv.org/content/10.1101/2022.06.03.494563v1) — Justas Dauparas et al., bioRxiv, 2022

## Practice objective

Design one sequence for a ubiquitin backbone. This does not predict its structure or validate its function.

## Requirements

Run the public model image with udocker on an x86_64 Linux Colab runtime. No Docker daemon or platform sign-in is required.

This small example can run on a CPU. A GPU is not required.

Compressed image: approximately 3.2 GiB. The preflight checks a 11.6 GiB free-space budget for download and extraction. Run one model notebook at a time.

Image: `ghcr.io/connects-scv/laidd-proteinmpnn@sha256:58cc21dac3a3168616fe6ffc10252d875c78bc1ffbd8df7cefc848a17f251ac5`

Platform catalog version: not recorded. This notebook uses the model and weights in the image digest above. Do not assume results match a platform running another version.

Runtime documentation: https://indigo-dc.github.io/udocker/user_manual.html

Installed environments, containers, and results may disappear when a session ends. Download the result ZIP.

## Procedure

Run the runtime definition, environment setup, inputs, inference, and result cells in order. The first setup downloads a large image and may take time.

## Runtime helper

In [ ]:
"""Shared runtime embedded in LAIDD Hub container-example notebooks.

Only setup() downloads software. Import, input preparation, and result validation
can be tested independently. Images are public and pinned by OCI manifest digest.
"""
import base64
import csv
import gzip
import hashlib
import io
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import sys
import time
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from sysconfig import get_path


class ColabModelExample:
    def __init__(self, spec, assets, work_root=None, execution_mode="F3"):
        if execution_mode not in {"P1", "P2", "F3"}:
            raise ValueError("execution_mode must be P1, P2 or F3")
        self.spec = spec
        self.assets = assets
        self.mode = execution_mode
        root = Path(work_root or (Path('/content') if Path('/content').is_dir() else Path.cwd()))
        self.root = root / ("laidd_" + spec["slug"])
        self.root.mkdir(parents=True, exist_ok=True)
        self.run_dir = self.root / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
        self.input_dir = self.run_dir / 'input'
        self.output_dir = self.run_dir / 'output'
        self.input_dir.mkdir(parents=True)
        self.output_dir.mkdir()
        self.env = dict(os.environ)
        self.env['UDOCKER_DIR'] = str(self.root / 'udocker')
        self.udocker = [str(Path(get_path('scripts')) / 'udocker')]
        if hasattr(os, 'geteuid') and os.geteuid() == 0:
            self.udocker.append('--allow-root')
        # udocker 1.3.17 accepts a registry digest in its tag field (repo:sha256:...).
        # Docker's repo@sha256 form is not accepted for fully-qualified repos by this release.
        self.image = spec['image'] + ':' + spec['digest']
        self.container = 'laidd-' + spec['slug'] + '-' + spec['digest'].split(':')[1][:12]
        self.parameters = None
        self.ready = False
        self.completed = False
        self.gpu_info = None
        self.image_metadata = None

    def _call(self, args, timeout=1800, capture=False):
        return subprocess.run(args, check=True, env=self.env, timeout=timeout,
                              text=True, stdout=subprocess.PIPE if capture else None,
                              stderr=subprocess.PIPE if capture else None)

    def _container_exists(self):
        if not Path(self.udocker[0]).is_file():
            return False
        return subprocess.run(self.udocker + ['inspect', self.container], env=self.env,
                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=30).returncode == 0

    def preflight(self, cached=False):
        if platform.system() != 'Linux' or platform.machine().lower() not in {'x86_64', 'amd64'}:
            raise RuntimeError('Use an x86_64 Linux runtime, such as Google Colab. macOS/ARM is not supported by these images.')
        required = (0 if cached else self.spec['compressed_bytes'] * 3) + 2 * 1024**3
        free = shutil.disk_usage(self.root).free
        print(f"Disk budget: {required / 1024**3:.1f} GiB; available: {free / 1024**3:.1f} GiB")
        if free < required:
            raise RuntimeError('Not enough free disk for compressed layers and image extraction. Use a runtime with more disk space.')
        if self.spec['gpu']:
            if not shutil.which('nvidia-smi'):
                raise RuntimeError('A GPU is required. Select a GPU runtime before downloading the image.')
            result = self._call(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader,nounits'], timeout=30, capture=True)
            self.gpu_info = result.stdout.strip()
            print('GPU:', self.gpu_info)
            device = next(csv.reader(io.StringIO(self.gpu_info)))
            capability = float(device[2])
            if capability < self.spec.get('min_compute_capability', 0):
                raise RuntimeError('This image requires an Ampere-or-newer GPU (compute capability >= 8.0), such as L4/A100. T4 is not supported.')
        return {'required_disk_bytes': required, 'free_disk_bytes': free, 'gpu': self.gpu_info}

    def setup(self):
        self.preflight(cached=self._container_exists())
        self._call([sys.executable, '-m', 'pip', 'install', 'udocker==1.3.17'])
        self._call(self.udocker + ['install'])
        exists = self._container_exists()
        if not exists:
            print('Downloading pinned image:', self.spec['image'] + '@' + self.spec['digest'], flush=True)
            self._call(self.udocker + ['pull', '--platform=linux/amd64', self.image], timeout=5400)
            self._call(self.udocker + ['create', '--name=' + self.container, self.image], timeout=5400)
        self._call(self.udocker + ['setup', '--execmode=' + self.mode, self.container], timeout=600)
        if self.spec['gpu']:
            self._call(self.udocker + ['setup', '--nvidia', self.container], timeout=600)
            if self.mode.startswith('F'):
                self._call(self.udocker + ['setup', '--execmode=' + self.mode, '--force', self.container], timeout=600)
            probe = "import ctypes; c=ctypes.CDLL('libcuda.so.1'); assert c.cuInit(0)==0, 'CUDA driver init failed'; n=ctypes.c_int(); assert c.cuDeviceGetCount(ctypes.byref(n))==0 and n.value>0, 'GPU is not visible inside the container'; print('Container CUDA devices:', n.value)"
            self._call(self.command(argv=['python3', '-c', probe]), timeout=180)
        self.image_metadata = self._call(self.udocker + ['inspect', self.container], capture=True).stdout
        self.ready = True
        print('Environment ready. Input files:', self.input_dir)

    @staticmethod
    def _sequence(value):
        sequence = re.sub(r'\s+', '', str(value)).upper()
        if not sequence or re.search('[^ACDEFGHIKLMNPQRSTVWY]', sequence):
            raise ValueError('Use a non-empty amino acid sequence containing standard one-letter codes.')
        return sequence

    def prepare_inputs(self, parameters):
        p = dict(parameters)
        for filename, key in self.spec['assets'].items():
            asset = self.assets[key]
            raw = gzip.decompress(base64.b64decode(asset['gzip_base64']))
            if hashlib.sha256(raw).hexdigest() != asset['sha256']:
                raise ValueError('Example asset checksum mismatch: ' + key)
            (self.input_dir / filename).write_bytes(raw)
        slug = self.spec['slug']
        if slug in {'anthroab', 'abodybuilder3', 'humatch'}:
            heavy, light = self._sequence(p['heavy']), self._sequence(p['light'])
            if slug == 'humatch':
                with (self.input_dir / 'antibodies.csv').open('w', newline='') as handle:
                    writer = csv.writer(handle); writer.writerow(['heavy', 'light']); writer.writerow([heavy, light])
            else:
                name = 'heavy_light.fasta' if slug == 'abodybuilder3' else 'antibodies.fasta'
                (self.input_dir / name).write_text(f'>example_VH\n{heavy}\n>example_VL\n{light}\n')
        elif slug == 'immuscope':
            peptide = self._sequence(p['peptide'])
            if not 9 <= len(peptide) <= 21 or not re.fullmatch(r'[A-Za-z0-9_]+', str(p['allele'])):
                raise ValueError('Use a 9–21 residue peptide and a valid allele identifier.')
            (self.input_dir / 'peptides.txt').write_text(peptide + ' ' + p['allele'] + '\n')
        elif slug == 'ligtmap':
            if '\n' in p['smiles'] or not p['smiles'].strip():
                raise ValueError('Enter one SMILES on a single line.')
            # This teaching example uses the human Kinase class.
            if p['target_class'] != 'Kinase':
                raise ValueError('This example uses target_class=Kinase.')
            (self.input_dir / 'input.smi').write_text(p['smiles'].strip() + '\n')
            (self.input_dir / 'target.lst').write_text('Kinase\n')
        elif slug in {'openfold3', 'protenix'}:
            sequence = self._sequence(p['sequence'])
            if len(sequence) > 150:
                raise ValueError('Use at most 150 residues for this small-runtime example.')
            value = ({'queries': {'ubiquitin_example': {'chains': [{'molecule_type': 'protein', 'chain_ids': ['A'], 'sequence': sequence}]}}}
                     if slug == 'openfold3' else [{'name': 'ubiquitin_example', 'sequences': [{'proteinChain': {'sequence': sequence, 'count': 1}}]}])
            (self.input_dir / ('query.json' if slug == 'openfold3' else 'input.json')).write_text(json.dumps(value, indent=2))
        elif slug == 'rfdiffusion-v3':
            length = p['length']
            if isinstance(length, bool) or not isinstance(length, int) or not 30 <= length <= 100:
                raise ValueError('Use a backbone length from 30 to 100 for this example.')
            (self.input_dir / 'design.json').write_text(json.dumps({'unconditional_example': {'length': f'{length}-{length}'}}))
        for filename in self.spec['required_inputs']:
            path = self.input_dir / filename
            if not path.is_file() or path.stat().st_size == 0:
                raise ValueError('Missing input: ' + filename)
        self.parameters = p
        print('Prepared inputs:', ', '.join(self.spec['required_inputs']))
        return self.input_dir

    def command(self, argv=None):
        return self.udocker + ['run', '--nobanner',
            '--volume=' + str(self.input_dir.resolve()) + ':/work/input',
            '--volume=' + str(self.output_dir.resolve()) + ':/work/output',
            '--env=OMP_NUM_THREADS=2', '--env=MKL_NUM_THREADS=2',
            self.container, *(self.spec['argv'] if argv is None else argv)]

    def run(self, timeout=3600):
        if not self.ready or self.parameters is None:
            raise RuntimeError('Run setup() and prepare_inputs() first.')
        self.completed = False
        log_path = self.run_dir / 'inference.log'
        started = time.monotonic()
        print('Running inference; log:', log_path, flush=True)
        with log_path.open('w') as writer, log_path.open('r') as reader:
            process = subprocess.Popen(self.command(), env=self.env, stdout=writer, stderr=subprocess.STDOUT, start_new_session=True)
            try:
                while process.poll() is None:
                    output = reader.read(8192)
                    if output:
                        print(output, end='', flush=True)
                    if time.monotonic() - started > timeout:
                        raise TimeoutError('Inference exceeded its time limit. See inference.log.')
                    time.sleep(1)
                print(reader.read(), end='', flush=True)
                if process.returncode:
                    raise subprocess.CalledProcessError(process.returncode, self.command())
            finally:
                if process.poll() is None:
                    try:
                        os.killpg(process.pid, signal.SIGTERM)
                    except ProcessLookupError:
                        pass
                    try:
                        process.wait(timeout=10)
                    except subprocess.TimeoutExpired:
                        os.killpg(process.pid, signal.SIGKILL)
                        process.wait()
        self.completed = True
        return self.validate_results()

    @staticmethod
    def validate_file(path):
        name = path.name.lower()
        text = gzip.decompress(path.read_bytes()).decode() if name.endswith('.gz') else path.read_text()
        if name.endswith(('.sdf', '.sdf.gz')):
            if '$$$$' not in text or 'M  END' not in text:
                raise ValueError('Invalid SDF output: ' + str(path))
        elif name.endswith('.pdb'):
            if not any(line.startswith('ATOM  ') for line in text.splitlines()):
                raise ValueError('No atoms in predicted PDB: ' + str(path))
        elif name.endswith('.cif'):
            if '_atom_site.' not in text:
                raise ValueError('No atom_site in predicted mmCIF: ' + str(path))
        elif name.endswith(('.fa', '.fasta')):
            records = text.split('>')[1:]
            if not records or not any(''.join(r.splitlines()[1:]).strip() for r in records):
                raise ValueError('Empty predicted sequences: ' + str(path))
        elif name.endswith('.csv'):
            delimiter = ';' if text.splitlines()[0].count(';') > text.splitlines()[0].count(',') else ','
            rows = list(csv.reader(io.StringIO(text), delimiter=delimiter))
            if len(rows) < 2 or not rows[0] or not any(any(cell.strip() for cell in row) for row in rows[1:]):
                raise ValueError('Empty prediction CSV: ' + str(path))
        return True

    def validate_results(self):
        files = sorted({p for pattern in self.spec['result_globs'] for p in self.output_dir.glob(pattern) if p.is_file() and p.stat().st_size})
        if not files:
            raise RuntimeError('Prediction output was not created. Check inference.log; logs/configuration files alone are not results.')
        for path in files:
            self.validate_file(path)
        if self.spec['slug'] == 'ligtmap':
            status = self.output_dir / 'Output/status.txt'
            if not status.exists() or ':Fail' in status.read_text():
                raise RuntimeError('LigTMap reported a failed calculation; see Output/status.txt.')
        return files

    def archive(self):
        if not self.completed:
            raise RuntimeError('Inference did not complete successfully.')
        files = self.validate_results()
        manifest = {'model': self.spec['slug'], 'image': self.spec['image'] + '@' + self.spec['digest'],
            'gpu': self.gpu_info, 'execution_mode': self.mode, 'host_python': platform.python_version(),
            'parameters': self.parameters, 'argv': self.spec['argv'], 'udocker_version': '1.3.17',
            'prepublication_colab_validation': 'not_performed',
            'inputs': [{'path': name, 'bytes': (self.input_dir / name).stat().st_size, 'sha256': hashlib.sha256((self.input_dir / name).read_bytes()).hexdigest()} for name in self.spec['required_inputs']],
            'reference_assets': {name: {'source': self.assets[key]['source'], 'sha256': self.assets[key]['sha256']} for name, key in self.spec['assets'].items()},
            'outputs': [{'path': str(p.relative_to(self.output_dir)), 'bytes': p.stat().st_size, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest()} for p in files]}
        (self.run_dir / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
        (self.run_dir / 'image_metadata.json').write_text(self.image_metadata or '{}')
        archive = self.run_dir / (self.spec['slug'] + '_results.zip')
        with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
            for path in self.output_dir.rglob('*'):
                if path.is_file():
                    bundle.write(path, str(Path('output') / path.relative_to(self.output_dir)))
            for name in ['run_manifest.json', 'image_metadata.json', 'inference.log']:
                bundle.write(self.run_dir / name, name)
        print('Validated prediction files:', [str(p.relative_to(self.output_dir)) for p in files])
        print('Result archive:', archive)
        return archive


In [ ]:
SPEC = json.loads("{\"slug\":\"proteinmpnn\",\"image\":\"ghcr.io/connects-scv/laidd-proteinmpnn\",\"digest\":\"sha256:58cc21dac3a3168616fe6ffc10252d875c78bc1ffbd8df7cefc848a17f251ac5\",\"compressed_bytes\":3452803636,\"gpu\":false,\"argv\":[\"python\",\"protein_mpnn_run.py\",\"--pdb_path\",\"/work/input/input.pdb\",\"--out_folder\",\"/work/output\",\"--num_seq_per_target\",\"1\",\"--sampling_temp\",\"0.1\",\"--seed\",\"37\"],\"parameters\":{},\"assets\":{\"input.pdb\":\"ubiquitin.pdb\"},\"required_inputs\":[\"input.pdb\"],\"result_globs\":[\"seqs/*.fa\"]}")
EXAMPLE_ASSETS = json.loads("{\"ubiquitin.pdb\":{\"gzip_base64\":\"H4sIAAAAAAAC/5V9W5JkN47lf68iNjBhfD8+1ak0qc3UirKSZmyq9r+Q4QFAEnB3eMfoJyqlyhO8JAjicQD88vfXf3/QP/Hj48/1479//v3xC/8R/6T+mUtYP8tnyQH/4rPFsv7zZ1h/mp+tf9x/1t//j18OYPr4+PHLM2D7TK2tn/WzxEyAoyQBjOEzDwX4wwBm+hcvACeA1s+WJ/5vnzVHtcLkAq7v+Hr5yWM0AiwNKyxrxdkB/DKAdf2G/3wGrJ8xJtrDMQKtsJW5Pzl/9u6ucC3jx2+vAHPNDNgAuH5B3ocS1wZPF3D9qr9+fQbMnxOni591EuAMB7B/Rr3CvwzgOq4fP58Bl7iUzoCj4PeubQt3hTG6K5wsh7/98ScBpnPKOVc6nN55D1Md91CSK4cxsBw+A46KT56fIUUCHFMBBvdQsPYfL1cYAwMmHM7awxS2YH+Mz+7KIWTj6wUgTrfKCgsB5hwuYHLlENcBcvi8wp7Xf8rhM5ZBV2+Gug9lnVhzV1hYDl/s4Vj/O8fPSn85f5Ywr9gE95QjbsqvL1eIq5fTZ6VrltbB7k9Oa9nBBVx/6+tnfAJc4jLwyfkz0k2Jn+MCrj9Mdw/X+f/5Mz0BVl7ZWiHtJQBD2588P8t05XCwYP/XHz8JMN89xF9ap9wq9nBdX/xuPuW6lusCThbsZ8CEm7JOOeCGrKtXpgL0BRsX9MfrFcopl8SAox3BLm9OGffq6wVgX3LXCLCTIuhLwZYL2KJ3KBB5CPbzCnMpfFNwp9flaGUL9trQWt0V4qb8Fl+scEDW0rq2dFPG55zzrtC/y4luSnoCXK/djKK+IgGmoQ6luq9eopvyaoUTb+9aYSCgsJ7Tfp/RkV3AxnL4j98ZsFyxwTVfgh0gLtBYPd5PNnto5BCqF3L4DNixT+vqlQzBnp/xyuE6Zf+TB8vhI+B6h2okwEzyF9aNCXcPs6sc8B5+vfzkDADc5Q7VHz/zrM6hGDnEr4IcPgLWzwaAvHTpFLEp7X6yfygwWKBgnwHTGATYoFjpLh85XCLg3uWcWGweActS/Z32cMIKWxghj3tTmms5QNn9+DW92MOI081r/wurr1bUI5WLC4ib8vN5hesdxoO4PjlgV9Y5tN7vu+w/AVB6P36mF3tIdxkrnInf5aZuSnPVF77rx78/Xuwh2TJrhSQ2ONia7ydPf4Wdr97/+eUPApQrte5wwuOEQymZ5DBMJTaleFcPR4er9wzY8LDTKUOg13MazgqzfVPsCidfvReA+F1rDwv2K8JyCHeF0xVsyMLXC8D5WaHMM4ymTID52IcfeBK9q1ciX71nwNn4oc+kB5dbcf2UtVxXsOF84Al4AFzPZ4qibSaAylIw5SqH5KqvkvkJeLHC2unVG3Tl1ifXpATb38PCYvPHv/4iwHZtG9KHS8BJoJcnUdQeBlds8DZCbJ4BGx6pvHw+GGgRyiJfwOi+erj5P16scCxxmQRYOgCX+OhTrq4cQkd9vQRM2KclNp2u3HLRYryH0psrNoPF5hEQrx6vsI19yv0q2OJaDrDZoLEfAddhACjjQCcBtnhs7PhmD/ESwSR+BFwrwuIXYCKlsACvv7zeZVfBQjbgmj0CZnYrSlhmTCbAfr3RYa+eBVy/9s9/PwMuQe4bMBDgPHu4jHbfcsARQrD//v2fBNjPTSmwUvLgmAPE5j6j61CiC1hYsB8AcZdhR2XYhSw27VqwC9C9y1j7j5crbI1PedApL4zjgMNycsUGJv7XC8DB3ujaw0oaG97pvIC+KYI7BMF+/ORlW0NvrE8efX5wVGCqIIb76uFXfS19+AIQ79daYcLvXIcyUrmC3f0VTtaHj4BJXj1YspMAY1dxm+gqB/wt0oc//zcBjnPKEWbPWmEgwKFuygIcrj7EZpA+tIA4ZQhyWdcWcRu4d9dyWMrRPWUI14+XKwx0U9Z7TE/A4OjI9pez/8lZ9OHTCiv85ZKXLpUVjuw44OZQIL2kD58AJ/Th2sMBO2Zd33nsQ1wb95GC1JI+fAKM5IDPpRwQA4trxVFpG/fqwTyFBfsMmAPrw145ZBWO0b6+P7uCjaODBfsIuJ5RMouwwkgrTGncFVbXcmjDqq95TnniGcUpk0AvAZ8KMGdXDqdVX/NcvYK9K7APAwHWY3CuT06uxu7Bqq+7wg7NVtbbglcCyuHY2CRDLmC06uuuMNFNgW096C7XtjX2OqHomnM9WfU1j3KYUFv4ZAROoGCPR49r496Unq362oBrEb0QYKA7jDB0vY9U8ldYrPq6gA16sMDgFG3T093D7noB0AAc4fwXALdDk0T+FmDDR0DBDiXYxRUbWD0c4bSA2KbBe9j4XW5dyWH3xabvCOfjClvgPayBDc48lbbxI+34tV8vAcn6wgoHuxU5RLWH/qFMYxLvcHLG85np6jWybfLyl+uNtNfm7eEIxiQ+gEv1y10OMN2wwpNcWK/edLXNiMYkVisslRVsoxuS1qMVrjca3as3kjGJFSCFm5f6Iq2zvjLm7PjLZg9HNibxAUwsyAXRuS7Oo9rD4j4BoxiT+ADm5SwFAgx4rCIUbFdviqtgRzUm8QFcFmvjh76TTbP04wnqQtu6cohfr0ziA1g5qYVXr3AwLRUVdh7+KXdjEitAygXAFMFdpnOIN+w83bs87Juy3QXYNiTYYcnfoBVSjuqYcz6gfVM2ICwHGEcZyYUgwTQlNsO1baZ9U9QKy2SDM5FxCXU2701pbnJh2jdFAVJgPItpDLP6RtqTfUaNYE/7ppxPHp8x8k0J5PCsw7muWXj3yfZNiTe5QIK9bZuIh79dwOlaX9O+KQqwUrhv6VJEs7CHZ4WPp2xXWE1yYccS8NBTQLLLpwZtLK0TS57YzGaSCwcwso+3HPBEuRPY6e2esu88zm6SC2qFE6bIujEUblmnXI7jE8Ob0P0cJrmgAEPiGGwjZYrTDkpju1kzqBKVXDiACB7xCsfguzxvshU5OmeFpAFUcuEArmeT0h/L+hp8U8ZdYfGjc6RCVHJBAUZEh9ehECD0Ya7KCxguYDLJhQNYln/CexgSa5uus7fF/2Tr0W+7D/YheaNlPfQcdla5gGW0e1mzGKxHfwADP0rr0yncgmBSTN+4KTFYj16tkFKXeP0oR99UzAEvVnABrUd/AEGUYECKZ1P8TAWCmqe+YrAevfrkQNG5wlYXpfLGN4gTMViPXn9y4Th2jcxzKEO9yzm4K7Qe/QakuA1H2jtZWuA9fOcJiNF69PHGYDmYtlZK2rmJ0bQz4K7YROvRH8DIyQQYTV0As4qKBFdsovXoFWAgt3ZyogbZiuNWfLxxHukl+noGXEAjcDqYclFIQ91U5nJVvYR1jNajVyukhDWMd7p6Y+lDJYe+cojWoz+AS7BhAuO0KTOxtq0d5VDtCi2g9egPYOZQ/TqUTLkTnIMCHP4KrUevPnmWKivkh54ykBswuQp25+h/+4MB2wEcUPmIZ0dOIZm77Eba487RPwMm3BRiV0WmGHUFWF1ts3P0z4B0l1NbSoEJPGFot8IV7J2jfwRE8IJpWiPwJ3PmZ9uHrvraOfoXe4iNh9Ek1I4xVewrJXeFeZNPHgEzlCgdClM75gnqpmztQwtYNvnkEbAhvoEkA5lugbO4DNj8FBLJApNPLOBSX3BukD3DNv+vJeA3KjKs5WD3kNgs6cWhDCFBjcYrTOcJSA+ZRwtoM4/xBsYz8RzW6VZWX3moOPZ0FWyymUcFSHowLYEurGDjVLlR18aOyWYe4420Jwj0OlByGqFgS7yAwTM4yav+egG4xAV7tgApA0lsjup8stnDbDOPaoUzMBmv0gF0/QRMGxi3K7SZR7WHxK8hzghr7DzmdXyCu4fZZh7NJ/NdjuRWIOKpvIDq72Gx+lDFYKFdEhbTmbOUVT7Ff5dztfpwqETh4HBzYULjPOrr450Fm5vVh0MReCYBbuJEju1mHou/h93qQ7XCIoAUllqH0oKTDrZiM6w+HCr2lSlk3yYLdg3KeUz+J0+rD4eKHw4CpOAumFn3XV6PtvvJJVh9eD+5QzQS8imR6YLX4HyT/oi4oFof3kMhmmBEqIqfgDkU/9D16CP8VK0PlRxCABI8WSbjKdrqsCkkCyh+yj/++UWA86SDa+VTpvcZsjyGEmz3oS/ipzwCIks26ZMHtAzc2/PqQbBdU6SIn/JihaT/1uHgyoH6WqMi8PiHIn7KM2CmeOEyVslIQoCyX8Ds76H4KY+AhZP+yMXQCvvy8JVycDOPxEklEtQTIKUuwf/M7Ov1oB6p6LoVlKP/9dWh9MiCTb4RuBI9q3Sw+9BX8VP++kmOz3YKYcY1yfSQxT9YjA6bJXhiU8VPeQRcDxuuHtLAEghKUREnfOVQxU95BJxCl4Ye5KjI1A+9rw+r+CnPgMQYRzSgc8i0BZWjT65tU8VPef7kmiTtkTjcF4eiT7tckQhl9/XbK0Am3zVOHa1tmzd7G61JbFcoDMlf/voHAcYbMp2c6aGfsIKTUg7VP2VhSD4AUnSO9WGgrG1QWTN6pN1PFobkMyCxTBdgbJytuLGvt89oFYbk8ydn2rMph5PWXirrK3oRTiJb45QfV7j2HecFWsgOSJahPCn3k5swJJ8BJ1aWxMcj8kW9vl523Qps0NfyRh8B1ydOfpezPPSltvumBFdsKEe/vNHnFRLfKyHCJPqwqWc0uzelFRNM2x4SPHpSBkvAM6eQxlWw1WpsI4etmmCaAsyTLYfQmASVhjKWXIJ3bM0E0/QKJx8KJ6rLepeUjf0GsJtg2gFMnySiaw9z5CTXaN+Tw2GCaQqwwASBkTB5hXNqL8AXm2mCaQcwC4cY7zLzvmrU5pwXdo6Uo7/BNLVCpgc2pidgD2/1x3zz0PdocgHphu4p6IR8VhNF2zRT18sFxJ5MLkABsq+XPlOIQsZTQV1/D3s2uYB0I+2d/JS09o45S7Urpq4vNr2YXEC6gfEE2yYhV8+Axr3trtj0anIB6SYXamLHJwdmYvRTuYAouatgKUd/cwEbEBULgT85EDMNPId09aEf1KUc/c0FqBUOnCQ+WR6rXFWU2CUnR6xd5QLOCpf+g5W6VjhJUzcVxKCEvQs4jfO4bed1KLWwPqyJ3+dZlWBX18YewTiPB7AyUzxFScN14X/tnJQrNiMa51EBEmdzfXKczD/sVQFG118eyTiPB3AJshxKIJNkASYlNm5FYRzZOI96hVT9gbD/Fpv+DUZQHMU4jwewf1JIB47PZMBx2CxgSLk29qjGeVSAtbGfMir/DMecQ0rNVQ6jGedRAfbKCpaq4VCVeDR2LtbGtnvYjfOoAFsa15wj7tj2AnBirnLYOfpf/qLSsFRv0j/zI9USP6OKZfqGwBN3jv4ZkCJKSeLZ4NsEVSo73UPZOfpnwJQ59pUH24cz6TfF1TY7R/8IuIwDKuYtnIGktLxiEfge/c7RP6+QTBAEgiqTT9I159obBTvztg+f9rCwcihRiBOHZQo17ioHuOlsHz5+MpU10Smz0d5OBhwFQK45Rzl6sg8tYGE5hD6c7OvdQ1kb2123YjYT1N0uF7K2gWtvu4hNqpp17z70s5ug7gFERXUmv5lKw6IOBCFu5Uba5zBBXbVCciOQraCrl4V9vwteXOUwpwnqKkDKzS9NFUhjgxSqCl6KpxxSCCaom26knYJnKMCqQnW7JYrZT/onztHHJ8DIn4iquMKfnIsOmVYXMJmgrgIkZx/hlsTcuXLrpIZfaJB2jl44SzsbBlYVJbnggCc2iU9dAMr+vXc57Rz9IyDUF9fRM4tlvXpZlTW5z2jaOfpHwGUpwDBDQWriZzSG7xRRpp2jfwRcjxQlF8AZE8CqyjxdtyLtHP3zJ9Onoqwk8DMazzO6xMalraYwDHdOHUou3NyA+a9dV6m/qU9JYRrunAKslF9GzIs9exXHBhnMA6Q6+p/PgEu7BD5lcs0QVEvp2oeuwUl0H8WdU4CFQgT1c0Z+/WpR2dviubcJIk/v8h+/EOA4p0xFkwj7Ffb1atch0+gCZnmXHwCX5zD5pvQm3qjOVrjEWiqw/fHxEpCegKWcJ9/lFr9TCpEgbF8vACeHSCm/zCECRfCOVmyMYFOO/j9fASZYeqhGojucbajKtRyoll1rm3mtr5wlns01j6Mp9TWbeyjDapsLWDpXWGfaQxBs1U15I4fTahsFWDgdTOXviE7dMEvzLVjKHHy9ABxMtUSipjNdMA8FOLxIe0rRapt59CGZPXj1qgR1j9i8SyElytH/9hKw8kNPFa4wWm8/B7wHLmC22mbeNwUrozpmDureulHKfLmAxWobBTg52UotA3Dbjtik5PO+qP5La5sLOCkXgK8rwvuK1/qaw5PDXUcv/vL2g9cejsj0GArtgOA9VbeE6qqvXUf/DNhhpSL2IICqucG7V2/X0T8CDm7/AerlYPV1/eXH0L0FnCbMogBrKRKlCxKQVOTk4pnEadfRPwIuxRp4hRQQgoK9yYVqQ6ZmhTmaMMsBLFzTswA5iLaLNnZ1sCvYOZkwi1rh6PLJxMcumn+YfD8lcR398wpBki9CbGTnsVYF6AbG083Rk1uxHRrQpoc0e4n8yT2qJFdwFezN0RtAFP5RMS9WyJ8cowqZNveRujl6C4gy98QhqzBEbOI3QlXp5ugfASkhg4i7ALYRHVKolcNhGuaoTyZCN5HxOKg7NLXDt7Fvjt4CJjbjwPIrXCd1k1yIH7qCfXP0FjCzw4NHKnF0LuiCF18Ob47+cYXUXwm14JWN9nlpCcWWhpk9hDejGuaoFZLFShlIic5VxXburhzuHL3kU3Z8GmITWMEO6doxsxab7AIWk4ZTgD0OYY5LX5FT9h7f9LdJO0f/DEjigrxKlndZ1426nKW0c/SPgLtUdukBaaWSbxFltvRpeyjdpOHMCgdTPAY/9OPkl+N498nDpOHUChveDWRxK3ePmSc3mopPxktwMVUa7gBmpvpSbwzmOdTD1E0PLc7MJ0O4VBpOrZCKyXFTIqfh+l1h9bkiRK9QFuz+lALFyv0cqBCVUuoqr5e8MAtdA2XBbkAUk9Orh4IDJtamrOXQ1Yc1GwtWA0JV4vXrXGhwk1wI1bkauxZjwepPprJ35FW5FCIXBfhmD6uxYM8KB0faQWwMXP1x24CgotJVsLUZC/YAdibuAJAOYDDrdMvhmz3sxoI9gI1LZFHuSYyBLt07JDDuslmIbqEsWA2YuIHYMTiPawZAVzkgMqEs2AMIL3RI9xhmjJdDF4ROcxXsrqOXwPhu1ZMnq37EHqiMJLLvd7jErr+86+hfAPYmsViOH4auXj2XBJV2Hf0DIIonO5fk1MTRuRRUEMNN+qddR/8MyBR0REcHASpPKvodydKuo3/8ZHCIuRSC29As/TiVcuj+J1eToMkqQVN5D5kzh3iiLuZ1A0FcR/8SkKyvzV1aSEfbIKXrWg6Uo78JGgVIWdu8mZFR+SlIYbgxB0ivStDkmzWjJD9WKN7oOJElaGz3CWjTVFjnel+92G8FDRWz6hisq7F7MBXWBzCx/oOCbWwflqwiS24fTlLHP16ukGKuCOF3Dkje5qiIObjPaE+mwlqtkPiHlBNgulbP02mlYvawZ+vetnv1Cnd1m5GztzEq8slwtU0v1r29gORKg0BBgJ0DlId84t6UXq17ewGJqYv+Ip0pb1M3A/Sdx96se9uO+iKeDZlzTTh0CjC6JnHv1r1t99XDSWZkzeSTp6rxKf4nD+veXkCKfZGCZcBb4wPunKsP+7TubVMhAj6USTcEgWHVFsnfQ8rR//q8QiT5uYFYF1JeG0pju5TLNKJh6m6bhXqy8CkH0oeTSQDHG/WaDqWRDFNXAVKBC/XGYA5dnarRsVuYn0Y2TN0DCGNpuxXCeo7jGxXWFAf4egZEDLZxiIDyymBXJfXJbsKaXm/F1D0rTJxCRyqpMOA8IQJEiV2xGc0wdRUg8a/BrhKCbdVF5W7hFfXCVExddSik8lHhOpihNqIqJ/Hdip2j34DjHEqoU7jtDNhCdxhBVmymFZsbuh9R+nAm7sAzhq4odMVm5+ifASmfB0ChyeSighj+u7xz9I+AnQsNAEhk+ajp083SVo3Y7Bz9I+CUmMO6w8Jtz0nZh25KPe0c/YtDwa1EjRWtDKQ8xVkaPmCxYjNuiEDqpGZgEhQ3wdoVNK4XsOvot/M4zycTGRR7OJn/lXRRefJy9GnX0T8DBile45+ZO4eeSLurYHcd/SMgrK4qJRGcqLmpTJizroLddfSPgNIRj2JfkglPKubQXXNu19E/AKJ1RZHI0pTEdVMhApfGn7mO/hVglbtcBycM2ymiRGGuDxhtzOECkglyanCrov7ibfU+mVg5OuagVlimEpumKv1z8UmheefoJZhWbqS9RI7B5s5ic5UDDG5PDvPO0T8C9mUPcsi0ZeGMxO8EJPPO0T8CopykSP85obyN/g3qb945+kfAte8Saad+Dgvwtjh7TBTaPewmqKv2kHw8hAiovCnrGOwbdl/eOfpHQOmZpgBtP4fqAk4T1N2AkMMWpAVu41Oe5RvN86mWUwV1D2Bk4gR5AbyH7fAPyYzw9pBy9Deoq1ZIVWOgawk5tHfVnit6XkDeOfoNGC+RjCgSgUnJoF729A3OUt45+kfAIoINvldl/lf/TgvcvHP0j4B5aRm+emR4YoVD9cd2y97zztG/ApQBBIEBVW/nZjsn20NpVrDvJw8KsyyPKnep21N8G7eFBTnFWrDVocR0DwXE0mtw5neAwwq2Atw5qVmEZVpVf2x/D6cV7NtniY0k8B04uXA7kqEq0zOJKaCgBTuq+CGLDRVhozA9q6xZ9JILOYmf8ss/fyNAVWjQJdka+aakm2wFI8oFFD/lETByrAtPQGKx6UOx+1yKEUVsf7xYYeDaMsrrCX21KjaLW+aZk/gpD4AoDetDegWxwZl0ftklhZIVSZbD0wqpLRf2kIjeyDyOa4qk7K5Q/JTnFRYRbCrAotLtcUNVyd9D8VOeASmkgxBBZJZfONUfaJHuvikQtj9/PgPKzcgSsiL+V7706eQqWKqj//crQKo3hzwm4d30ck0RN2RKI0f+/D2+AKRUOvVB3NNy4jWWsmuK4Aj//D09Ae4+7UnKPRtXuG4+ttsjiKpMVfeYcmsr2mQLtojjw0WVewBBdQGz6R5TbilEl1YqtXOJ2M2NYuyQ+y7nYrrHlDvEIUsrFT7loFmm1bYrtIDVdI9RgNQkKg3mfeGT9cwFN2RKgS3VPUbtIdWoooUFpYN1ed1H9qluOXfTPabc+pQSOa83I2vsqdXXGzkcpnuMAqSHntwKoVwmxbp3G4hlytHf7jEKkAioWeI3BKje5eKecgkmZFruiASSP3qXOcwSgpLD6lGMyLRXIdNyJxqEzIEgohih9jGoLuj+J5dkQqZFTYUIUQYDsUcfp+LbuP2+iJv69Rpw8idz0ZAZd/LQmt7IYSkmZKoAqVETtcKVN0WXKA7XFKEc/Q2Zqj1spUnTFzY4S4rfCGLQvVIh03LnVpA9CI0tz+jtj41Xz7VtKEf/66sVdrEcyvYCbiFq9ukx1C5eTXjZZE/qwyvMtMHU31RUtqK7+rBMM+HlACY2kmDBNimmvK5Z8euXaerKjxcrDNLfZkhkaah+Do8msQWMZsKLBkwydod69k3V6x6C7Xr0NZkJL+qTqX8DFeh3Vg5BAfpPQM1mwssBjDIorXOvIBB4krp6c7iAxUx4UYBkRxHPgcs8Q1DZCrdyIVOO/k54UXtIsz/oPebW9KFmZ0CVBWxmwosCzLstUmEiWQiaj+1/cjcTXjYggridjSTmiKxPNj0xfLEZZsKLAqxCrC2NB2GEFJ3pJBZwGgr6VpxR+DV46KVKuHY1dqe6XkALhoJ+AAfbg7jLJasWAv+j0d6ioaBrwJqlawLT+JmKvuPY7l1uyVDQD2AXT74zmyVFrjA8YRb36rVsKOgGkDntkwgTYuO8bNxkV1hMOnin1/DK9SThPo60p6KL11x92KpJBytA4n0BkD45qbJ3Sti7K2wmHbwBYSkMPhTqpYsnIMXvhKpaN+lgtUJqEk0hU2Y916aCutN1zXaveyGfbBogFGxgBcuuWVKCjVYy3d3DaUhQBzAu84VDpeyamXbWYPN4n7x73T8CSs0tSPJD6pg1D9Y/lN3r/hkwS61Zo4Z2eqQgsXm8Pdy97p8/mciTGC0Y2aPP+ur5NnbPhgS1AfHKSQMxqvHBo1VVcsEXm14MCUoBtiyRzcHBtDtiNb4z2ilH//MZEBU07MlTRAlPwjkUSL2/wmZIUAewS/9D6TRBD3+/zQCnGz/s3QYk5zllqumBHCYOjF9yMuTQ9Ub7sAHJ23SI7CjkALoU6BetsX05nDYgqboYFS7WoBkgEOykGub4UZERbEBSrTBx6ogIZBD0rhLW3UsUEhdVByRva6kgXOIgwLOprJnLGKeyGB2QnLeBWOWrR8YFTnuqZi++chjZBiTn8eiJYwbgzkGMkFTj9+Sac5SjVwHJef3lMW+mZ4+t3XKYXAWLe6UDkveTe+8qyaUJPNSMzpPD0UxUZKd5KU4j2bIhfkpWQ75qdAG7iYooQKrGxFsi81NCTA7FyO7hMFGRertpscUA+joDqpsy3kTnxjRRkao6QYUiQ4n5XQ63iHK8ibTPYKIiaoXUrQMCPZkeoyoKu99tlfS6ioocwCrPJ/henAGfVYVMqw+YTFREHQoVoCaJuEepjjvVwa6CndlERQ5g4V4sxLvjXEDoiufg56Sm7SFZ75TtSa8eSiKGnPL4RrcEaousWKb19qqiHvcUv5ZI+1AGp+8vT9tDUgHyCiPTtOidVurL5SzlaXtIKsAi8Rr2V4Ls5WYEDVcObQ/JAzh4NhyM9SKzkG5vlvYmfjhtD0m1QioZwLiZytbX7RQKWfIAS7A9JNUKuScLuvCz+NwVwk72AW0PSbVCnqs3OJ6dNqFMAuNuVzfqvaVYpkoOuxRCD7HC8inWQC1ycQFtfUpVCRqqbI2S9I96okHyKUYl2PqUelNISfheTcSnxvKNUy7B1qcowCidQjncLKO0zjOaXUBbn6IAd2NPHr6JKpChejtHdw9tfUq93bQ45rAZGWIi7wSN2wK3BFufcgALj3+ikJU0Eju5AOi05ALa+pQDWLmcCca6zDWbVbVidttmUq9qxRVRn0xFlESY4J6m43j06cG2sYDRuLf1tueixrLwk6UwvwRVRx+8BA2lxpR7qwBr4xozbrQtPU3PBEDvXS4xG/e23kg7V3s0IcdPFbeBextcwGLcWwXIxLHKTwA+vekhDtXdw2qyt/VONMh74tos8uqpmIMbt6Ech8reHsDBLUdpYCSrsdRVqawv2LGb7K0CJCeLhsxx/HAMr47eAg6TvdWfTAFIlIYJU3cGh2Jk93Ca7G296Y8pgNRGCGS8mb4xw5pmSajsrQKkdupoviZ9Yc0gXbdoiAL8KnurAENkOcyiYNNQnZNdngNN5VPZW/3JMu+2ins7b1e3B9fMiA3l6P/9DNjXvnPriiLdLm9kCebscFdYTPZWAZbObWh65vhhzcpfzsNdYTXZWyWH3LMvy1x6ieNs9eU2Ry27jl76H+7kFVyyyFeP/BQUotbi3BQL2E2XSw3YN7uU+9vQrLj/cfZH2XX0j4DLkxqsB1lcii4nGX4Mtuw6+kfAys3K8YwOVmMqa1b9eVJl19E/fzIPfUXNLXeCqjk7MVizQjgjqsulWuFMnANo0ofTMMbdGp+y6+gfAXEZpNCFTJGsLYdojSW7wmzd2zvQmf3kwMTaaIIYze6hEZtcrHt7BzrXHb8OLOC3eO1xBo1dYbXu7R3onCTCXk77Vs2Q9A+lWff2DnROp2caH0pOaoSWW5VZcrfu7d3DMPiTx+AKaxMiaK4+zMO6t3ewOGdtweHkCYAzqCegu1ePc/TxBWAXMijzX4POp7yprShcR59eAPJEg8xDONG4SHesdVNI1CxEtQTf7T3olIUzl7lzcjLZCi8gSe0/VUvwA4jwHquvOhgwpPqNBA3liX58vAJMkmSlyngSGwVY/E8upiV4vdmK2biTPFUkU2F6/cY4PGqJoFqCH8DMxRpg9SUWm1hV20w37EyTu1RLcAVIzV7w0E8uolTNorOd1mRX2K2vN+6hTE4Dc0e8ogzOx6iIPeVhfb0LWGTeLTlAsGSnihK780ap5O3HixWirClIX2Keexv1q+c7PjVYX+9mfIq4FTyvoqmWPrgp7inXaH29m08p0jyfR6uiQ54KVbnxw1KT9fXGJZ8MPuUgJYpNjyFzhytRAlT7euMGxoN0TBbaaqy6ftl99aiOXvl695MrvSG7gV0Ve3G/eq6NXcVP+ftfLNjznPLYkzXohqBrR3VsbCOHVfyUZ0AK7+HVKzI+Xqc/3L5zNDvix4sVBqkxi9zTFJ9e1CPl8hxoLtHXMyBCVJOzZVxmXHVuNL0xRar4KY8rlEpC6EWZvXAnlWMPXX3YxE95BuQxZFHe5fVojeqYIhYw8qv3CIg+DrKHheWwFU0kc00RqqNfcvi8QrIBEBWpXRrA6xW6h0I5+p/PK6zcIy3tlgFDGjjtxk3+CgtTOx4BwTJnU2SGHTL16gIsYGVP6nkPa+xSD8C+3h02BxKU6y9Dr3/9/vHik9Ou0wsSLZ5KbNymQ6V100Z4W1WUo5c+7YM7QYWi4oe+a9aGaSOsAOlUKeYlXX+nLg3zBXuaNsIKsEjonhvMJnHVdn7Z3cMeTBvhAzg4f4KcaOQGYmmovnPuMBHKaag2wgewS04Ucpi40r+pweJubrT0ZNoIH8AmCcIoc1SSqgFPb4YrUdGJaiN8AKskrCMPqqJ2LKodjctMK/BhVRthtUKyDxEd3lO2LwV9yZCXsC7dzg7eDzjFXoUw0biFRdLZ2+DKYbezgw9g5HeZevdxy0cTWXJ7VZVuZwdrwBikHoC9gHhqfOIbcnLpdnawAhyF5ZDiiB8yM3PLoR/76nZ28AFMXEuR9jQIXdYEt8o1RYadHawBGxdPDgm3tKltG/fqDTs7+ADCP27Ch2WTWLV8zG/c22FnBx/AdRhJcqP0fBbVp/1dUTlV9Skmxg7jUUt65opQSQQ43adkO+Y3ts2wNY8KMAwG7OR5DtX5hEJ27gptzaMCpCp16ljLTSnzCQS9VbDD1jwewMj9lZBaz9w2k+r2Tp8lVzkMW/O4AXePe3xybtL8Ss0LcPPLZdiaxwO45+uhhQp/8vUCQHpw7/KwNY8HsEn1EcYy8sC+eNQXOh64xtK0NY8HsPIAZ6ivwaPcyjnl/IaJQROUFBNDrzBPGUeWqXFTODclv8vrzWS4c9sigL+cpOF2SwKoshVuaRjNJ1LcuQMYTykO1UuhAM6MPncj7bMY7pwCpMmhiNLJWMZbNAR2n38o1XDnNiD5J0PmPXKvKtXfZr5xwGcz3Dn9yZk7yVNbGpSkZF2l7gr27IY7pw6FquEQ3CURWU9BVwMIfJN4DsOd04BNejsTaUwOaRN4oqtgKUf/89UeUk94iA0FNpb2OQYn7qOnvir2WHHnDmBnQ5NaMRdaYTifDJ3mKdgaoqF27CAZ8WuYB0uxBsxs1fNG3XHJNSRD7TiAYv2jRFEAx9AtH4v7ydlQOxQg2fOoT6nc1S2171AuqTfr10tA4jUgtS7dBasehFG87C1NsVfUDg0I+aNJ0TycvQ812tKtT6GgjKJ2KEDuphV58ACmIpy7/K79Aj22itqhALkjWeAyz48ks7mE2uEGJKl7rKJ2bMAofBuqyswEGA+xlopJ3T2chtqhxaZKnRQ96qCt1qux3VBVjXbOY7u1FTT8FS19Ms+GU75es2FnI9jRznlUgLufA4VMKXupi3ndPYx2zmO79SkkoiQ2PO4k6pkL3VUO0c55bFWFqqSXaeIekrWqQRjT30M751HvYeY3Zcr8lFxVhNPtb0PzAlVQt91yEiZODKYYRWnxc6YbN3eFNvO4J7dQcxe+KaOzn6LqpB5a4NpTtpnHphI0kWtviZcNw3+qOvrhn7LNPCpAknkwgxKH7tMlkhU/QUMp2q+Xn8wdvKMkqgsL+InbeMYSTUNTmUcFSH4JdcbrnPG5sa/kJ2goX64yjwqQjApMy6FPDfycbkDXJK7JZh7VHpIJTFObOOPTU3JGxtgV2sxjuxmfPqVJb+I0XLv9vopNwxmxSTbzeAAlRAoiT+DugqpUNr/bQ5t5VIDEs0GjEuku2G5h/js5TDbzeAAxL4Bbj/bMGfAxdf1ycQ/FZh4VYN69CCbHvm7n5HckeeroojKPCpBuBqiXIwg9QU+86i6gzTweQOkwQfPA+aakWx38ZuxOzTbzqMSGWaZj7T/3Mr3jkhEicd/lLNbX7//F5tzNVhA9Jkv42c7Xe+h1b+Qwi/X1CLhOufNdJm8AYecaHMG2KxTr6xEQ5JMqrQOYEZSD1jb+Hor19QxIVgp4iKMLfXo4g3SNHGaxvp73kGa1YFpOkzj2UNPeq//JYn09AmbpiQGqIX9yTCpRON27jLfyzyWHz4ADz2ZuQjrpTJZ/GfuypzxYDh8BE9cr45NjEUA1x8ftb0NOMXIBz4BsLDWukwIPNOjG7+4nF+nN8gxIXS1zlXKmrsN9ze84UXeOfl+9W/1BqpLGkXWpX1bFvGm6gMlq7Fv9QdVwuCmFxWak+Y3xT3Xn6B8AYWhCGVCfm6fWAdVvjlp3jv4ZMFJzA2wXt83sQz1SbilE3Tn6Z8Ape8h3GGMaFVfEZabV0qzGnodLvHvqcjG5HmCKyJr/yd1q7AtYK/cY7wJo6vWGDzisxp6HqUtVtBCbyXm9VBRxwqVcUi84NbBvqyVqNiRXT5jjyq0ob+SwBjOwTwFSRBptWJpUFHZVR+9OeKk1moF9BxAOeJRpOdwt4TbPR1NKV2PXZAb2HUAZrYonYAozKKs9dIdiU/2XGtjXg6qt4JtCbRgQCNE1Pu6kIfJm1MA+BUitA9Bwm8pJMKpCrbD7e1jNwL5+q5Aoj0fqS4jeuiSnujGHaiu5+p1U3hENQY/xzJ8cZvhGoUGttpKr3wZicStYKVEsQ5V5Bv+TbSWXAqRJKJDDKjmB24dz2JyUBbSVXP32TOv00GPyFZeTXDlEYMwVm2YrufptIBZ34/fG9QHzJhe6T0GvzVZyKUBiO+OnECh6VpF2X301W8mlAEPgU2ZSXlDjQZHndANBzVZy9duEjbRM3gXR0pns5AL8FRZDku+3tmJmbldIZZGUvYy3WMOtAa+7jv4RcB2KKFgu2lhKeqoRq259St119I+AywTpDEgNWakWfKrpxu7V23X0j4Cdn88zYBzV6lVlK1xvdM+jf95DSrbSneaS2dvfBr6Ge1PaNCR5tYe9csyBml6h8KDpYJoL2IMhySvAJj3TqhSkhqIaiLm0VcrsK5K8AgwlSOwrSWVruA3E3PwyUacUSV7tITWJQthZah+vn4K4gSvY1Ov+kuQVIOeXh9BkOk/aOCtM7gqLIckrwBCLdNPqclP6XaEfad85+n2X77gT0oNQsLMLaT4jJbD0OCJs2QVs9glQA1ka+ykUjEfhARwfAsxv+sHWnaN/BJSKGVIS3HyI+rQzYHrj0e8c/fMKqQcyAAs3H6IZXRswuwHJnaN/sUJ8IlrUF25lQXVS+5Pd3GgdwT4BF5D2DkpCasAD/rwB/Uj7iPYJUIcyxJMqbIpQ6JRP+Z3GHsk+AXeF1J8NzcurykBuQLeYt+4c/RbsOz+FenHDJI7cUoVcNf7kbIvXjBzuHP0zIMVe8XdDugFyBqx+4/e6c/SPgJHNNzRJTVM6k5UN2N5YsDtH/wgo4RXI4ZTBfSNfQD/msHP0zyukaAj8lcTZs3vK+Q1dsO4c/QvAwH04ibJN8x7nBnwzXKnuHP0z4JCubnxjAlcJM+B849HPYJ+ADZilNHFya6md3yNATHhw1RfV0f/7GXBbDJP5r0myFRvQ30PK0asn4K6wSwqJWhej3O58MmbluRmfme0TsAErj3vCJ0vvtK4/2dfY03Yx6ndkTGwyCylL1gKODy4MufmuFzBtFyMFOAqHCMi/RKJstAvoxxym7WKkAIk7nDsrWIrSqRX6+nDaLkb6k0kpIH7DOVLKbW9A3wuYw+7hHRlDbwnNUeFxEx1K9Xyyf8rT7uEFpOY8WeZV0Gyuoj7Zs75aCHYPLyA9m+g1PtnNpbDf+eTgAka7h3fCy4QSRYv6xOLThvpkt46+UR39//37xQrpDcHr11l86NTPCh+rkP7++c8PBpRf9QD4//vPf/z889f/+H/hPFmjz74AAA==\",\"sha256\":\"65040ad65c3737da37511152dc972bb641210194aad9987eee6f303382818b64\",\"source\":\"https://files.rcsb.org/download/1UBQ.pdb\"}}")
EXECUTION_MODE = "F3"  # @param ["F3", "P1", "P2"]
example = ColabModelExample(SPEC, EXAMPLE_ASSETS, execution_mode=EXECUTION_MODE)
print('Model:', SPEC['slug'])
print('Input directory:', example.input_dir)
print('Output directory:', example.output_dir)

## Set up the environment

Check disk/GPU requirements, then prepare the pinned runtime and model image.

In [ ]:
example.setup()

## Set the inputs

Edit PARAMETERS below. To use another structure file, run this cell, replace the corresponding file in the printed input directory, and continue from the inference cell.

Required files: `input.pdb`

- ubiquitin.pdb: https://files.rcsb.org/download/1UBQ.pdb (SHA-256: 65040ad65c3737da37511152dc972bb641210194aad9987eee6f303382818b64)

In [ ]:
PARAMETERS = json.loads("{}")
example.prepare_inputs(PARAMETERS)

## Run

Run the actual inference command. Failures and timeouts stop execution. Every run uses a new folder to keep previous results separate.

In [ ]:
prediction_files = example.run(timeout=3600)

## Check and download results

Validate structure, sequence, or CSV content, then archive outputs, logs, image identity, and input provenance.

In [ ]:
archive = example.archive()
for path in prediction_files:
    print(path.relative_to(example.output_dir), path.stat().st_size, 'bytes')
from IPython.display import display, FileLink
display(FileLink(str(archive)))
DOWNLOAD_RESULTS = True  # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    try:
        from google.colab import files
    except ImportError:
        print('Local runtime: download the ZIP using the link above.')
    else:
        files.download(str(archive))

## Interpreting results

Design one sequence for a ubiquitin backbone. This does not predict its structure or validate its function.

Structures and scores are predictions. Consult the paper and model documentation for metric definitions. A successful small example does not reproduce the full paper or validate every platform input.